In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [17]:
stock_files = {
    "NVDA": "data/NVDA.csv",
    "MSFT": "data/MSFT.csv",
    "META": "data/META.csv",
    "AMZN": "data/AMZN.csv",
    "GOOGL": "data/GOOGL.csv",
}

news_files = {
    "NVDA": "data/NVIDIA2021_2025.xlsx",
    "MSFT": "data/MSFT2021-2025.xlsx",
    "META": "data/META2021-2025.xlsx",
    "AMZN": "data/AMA2021-2025.xlsx",
    "GOOGL": "data/GOO2021-2025.xlsx",
}

In [12]:
for ticker in stock_files:
    df = pd.read_csv(stock_files[ticker])

In [14]:
stock_dfs = {}

for ticker, path in stock_files.items():
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"], utc=True).dt.tz_localize(None).dt.normalize()
    stock_dfs[ticker] = df
    print(ticker, df.shape)

NVDA (1254, 10)
MSFT (1254, 10)
META (1254, 10)
AMZN (1254, 10)
GOOGL (1254, 10)


In [18]:
news_dfs = {}

for ticker, path in news_files.items():
    df = pd.read_excel(path)
    df["Date"] = pd.to_datetime(df["Date"]).dt.normalize()
    df["Ticker"] = ticker
    news_dfs[ticker] = df
    print(ticker, df.shape)

NVDA (2434, 4)
MSFT (2655, 4)
META (2411, 4)
AMZN (3982, 4)
GOOGL (1624, 4)


In [19]:
def shift_weekend_to_friday(df):
    df = df.copy()
    df["Date"] = df["Date"].apply(
        lambda d: d - pd.Timedelta(days=d.weekday() - 4) if d.weekday() >= 5 else d
    )
    return df

for ticker in news_dfs:
    news_dfs[ticker] = shift_weekend_to_friday(news_dfs[ticker])

In [22]:
merged_dfs = {}

for ticker in stock_files:
    stock = stock_dfs[ticker][["Date", "Close", "Daily_Return"]].copy()
    stock = stock.sort_values("Date").reset_index(drop=True)
    
    stock["Close_tomorrow"] = stock["Close"].shift(-1)
    stock["Return_tomorrow"] = stock["Daily_Return"].shift(-1)
    stock["Direction_tomorrow"] = (stock["Return_tomorrow"] > 0).astype(int)
    
    news = news_dfs[ticker][["Date", "Title", "Summary", "Ticker"]]  # 这行加上Ticker
    
    merged = pd.merge(news, stock, on="Date", how="inner")
    merged_dfs[ticker] = merged
    print(ticker, merged.shape)

NVDA (2403, 9)
MSFT (2640, 9)
META (2388, 9)
AMZN (3952, 9)
GOOGL (1613, 9)


In [23]:
all_news = pd.concat(merged_dfs.values(), ignore_index=True)
print(all_news.shape)
all_news.head()

(12996, 9)


,Date,Title,Summary,Ticker,Close,Daily_Return,Close_tomorrow,Return_tomorrow,Direction_tomorrow
0,2021-01-06,Nvidia's $40 Billion Deal For Arm Faces U.K. M...,Nvidia Corp.'s proposed $40 billion takeover o...,NVDA,12.579124,-0.058953,13.306580,0.057830,1
1,2021-01-22,Huawei's Honor Spinoff Unveils 5G Smartphones ...,Huawei Technologies Co.'s smartphone spinoff u...,NVDA,13.674047,-0.011177,13.614962,-0.004321,0
2,2021-02-04,"EU, U.K. to Probe Nvidia's $40 Billion Arm Acq...",The European Union and U.K. are preparing to l...,NVDA,13.625930,0.009885,13.552885,-0.005361,0
3,2021-02-12,Wedbush's Ives Sees 'Arms Race' in Chip Industry,"Dan Ives, Wedbush Securities managing director...",NVDA,14.919294,-0.018983,15.287259,0.024664,1
4,2021-02-12,"Google, Microsoft, Qualcomm Protest Nvidia's A...",Some of the world's largest technology compani...,NVDA,14.919294,-0.018983,15.287259,0.024664,1


In [25]:
all_news["Summary"] = all_news["Summary"].str.replace("\n", " ").str.replace("\r", " ")
all_news.to_csv("data/news_stock_merged.csv", index=False, encoding="utf-8-sig")
print("saved")

saved


## Data Preparation

We merge Bloomberg news data with daily stock prices for five AI stocks (NVDA, MSFT, META, AMZN, GOOGL). 

For each ticker, news articles are matched to trading days by date. Weekend news (Saturday and Sunday) is shifted to the preceding Friday, so that it is paired with the next trading day's return (Monday). The target variable `Direction_tomorrow` is the sign of the next trading day's return, where 1 = up and 0 = down.

The result is saved as `news_stock_merged.csv`.

## Method 1: Loughran-McDonald Dictionary

We use the Loughran-McDonald (LM) financial sentiment dictionary to score each news article. 
Each article receives a sentiment score based on the proportion of positive and negative words 
in the title and summary. The score is defined as (positive_count - negative_count) / total_words.

In [26]:
import pysentiment2 as ps
from sklearn.metrics import accuracy_score

lm = ps.LM()

def score_text(text):
    tokens = lm.tokenize(text)
    score = lm.get_score(tokens)
    return score["Positive"] - score["Negative"]

In [27]:
all_news["text"] = all_news["Title"].fillna("") + ". " + all_news["Summary"].fillna("")
all_news["lm_score"] = all_news["text"].apply(score_text)

print(f'Positive: {(all_news["lm_score"] > 0).sum()}')
print(f'Neutral:  {(all_news["lm_score"] == 0).sum()}')
print(f'Negative: {(all_news["lm_score"] < 0).sum()}')

Positive: 1995
Neutral:  5023
Negative: 5978


In [28]:
daily_lm = all_news.groupby(["Date", "Ticker"]).agg(
    lm_score=("lm_score", "sum"),
    article_count=("lm_score", "count"),
    Direction_tomorrow=("Direction_tomorrow", "first"),
    Return_tomorrow=("Return_tomorrow", "first")
).reset_index()

print(daily_lm.shape)
daily_lm.head()

(4064, 6)


,Date,Ticker,lm_score,article_count,Direction_tomorrow,Return_tomorrow
0,2021-01-04,AMZN,-3,3,1,0.010004
1,2021-01-04,GOOGL,-3,1,1,0.008064
2,2021-01-05,AMZN,0,2,0,-0.024897
3,2021-01-05,GOOGL,-4,1,0,-0.009868
4,2021-01-06,AMZN,-1,3,1,0.007577


In [29]:
test = daily_lm[daily_lm["Date"] >= "2025-01-01"].copy()
test["pred"] = (test["lm_score"] > 0).astype(int)

for ticker in ["NVDA", "MSFT", "META", "AMZN", "GOOGL"]:
    df = test[test["Ticker"] == ticker]
    acc = accuracy_score(df["Direction_tomorrow"], df["pred"])
    baseline = df["Direction_tomorrow"].mean()
    print(f"{ticker} | Days: {len(df)} | Accuracy: {acc:.1%} | Baseline: {baseline:.1%}")

NVDA | Days: 171 | Accuracy: 45.0% | Baseline: 55.6%
MSFT | Days: 194 | Accuracy: 42.8% | Baseline: 56.2%
META | Days: 216 | Accuracy: 50.5% | Baseline: 52.3%
AMZN | Days: 211 | Accuracy: 48.8% | Baseline: 49.8%
GOOGL | Days: 194 | Accuracy: 44.3% | Baseline: 55.2%


In [30]:
portfolio = pd.concat(stock_dfs.values(), ignore_index=True)
portfolio = portfolio.groupby("Date")["Daily_Return"].mean().reset_index()
portfolio.columns = ["Date", "portfolio_return"]
portfolio["portfolio_direction"] = (portfolio["portfolio_return"].shift(-1) > 0).astype(int)
portfolio = portfolio.dropna(subset=["portfolio_direction"])

print(portfolio.shape)
portfolio.head()

(1254, 3)


,Date,portfolio_return,portfolio_direction
0,2021-01-04,NaN,1
1,2021-01-05,0.009758,0
2,2021-01-06,-0.029583,1
3,2021-01-07,0.028871,1
4,2021-01-08,0.003287,0


In [31]:
daily_port_news = all_news.groupby("Date").agg(
    lm_score=("lm_score", "sum"),
    article_count=("lm_score", "count")
).reset_index()

daily_port_news = pd.merge(daily_port_news, portfolio[["Date", "portfolio_direction"]], on="Date", how="inner")

test_port = daily_port_news[
    (daily_port_news["Date"] >= "2025-01-01") & 
    (daily_port_news["article_count"] >= 5)
].copy()

test_port["pred"] = (test_port["lm_score"] > 0).astype(int)

acc = accuracy_score(test_port["portfolio_direction"], test_port["pred"])
baseline = test_port["portfolio_direction"].mean()
print(f"Portfolio | Days: {len(test_port)} | Accuracy: {acc:.1%} | Baseline: {baseline:.1%}")

Portfolio | Days: 234 | Accuracy: 45.3% | Baseline: 53.8%


## Summary

The LM dictionary assigns fixed positive/negative scores to individual words
with no contextual understanding. Each article receives a net sentiment score
(positive minus negative word count), aggregated by sum across all articles
per ticker-date. Days with sum greater than zero are predicted as Up.

All five tickers and the equal-weight portfolio fall below their respective
baselines in the 2025 out-of-sample window, confirming that the LM dictionary
has no predictive power for AI-sector news. This motivates the transition to
learned text representations in Methods 2 through 4.

## Method 2: TF-IDF + SVM (Expert Committee)

TF-IDF converts each article title into a weighted term frequency vector,
capturing word importance relative to the full corpus. A LinearSVC classifier
is trained on these vectors to predict next-day return direction.

To enable a clean ablation against Method 4, we apply the same expert committee
structure. Each ticker-category pair trains an independent SVM. Within each
ticker, category-level predictions are aggregated by training accuracy weights
into a single vote. Final portfolio direction is determined by weighted majority
voting across the three tickers (NVDA, MSFT, META). Days where the winning
probability falls below 0.5 are treated as no-signal and excluded.

Train period: 2021-01-01 to 2024-12-31. Test period: 2025-01-01 onwards.

In [4]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score

all_news = pd.read_csv("data/news_stock_merged.csv", encoding="utf-8-sig")
all_news["Date"] = pd.to_datetime(all_news["Date"]).dt.normalize()

print(all_news.shape)
print(all_news.columns.tolist())
all_news.head()

(12996, 9)
['Date', 'Title', 'Summary', 'Ticker', 'Close', 'Daily_Return', 'Close_tomorrow', 'Return_tomorrow', 'Direction_tomorrow']


,Date,Title,Summary,Ticker,Close,Daily_Return,Close_tomorrow,Return_tomorrow,Direction_tomorrow
0,2021-01-06,Nvidia's $40 Billion Deal For Arm Faces U.K. M...,Nvidia Corp.'s proposed $40 billion takeover o...,NVDA,12.579124,-0.058953,13.306580,0.057830,1
1,2021-01-22,Huawei's Honor Spinoff Unveils 5G Smartphones ...,Huawei Technologies Co.'s smartphone spinoff u...,NVDA,13.674047,-0.011177,13.614962,-0.004321,0
2,2021-02-04,"EU, U.K. to Probe Nvidia's $40 Billion Arm Acq...",The European Union and U.K. are preparing to l...,NVDA,13.625930,0.009885,13.552885,-0.005361,0
3,2021-02-12,Wedbush's Ives Sees 'Arms Race' in Chip Industry,"Dan Ives, Wedbush Securities managing director...",NVDA,14.919294,-0.018983,15.287259,0.024664,1
4,2021-02-12,"Google, Microsoft, Qualcomm Protest Nvidia's A...",Some of the world's largest technology compani...,NVDA,14.919294,-0.018983,15.287259,0.024664,1


In [5]:
keywords = {
    "chip": [
        "chip", "semiconductor", "gpu", "h100", "h200", "blackwell", "hopper",
        "cuda", "wafer", "foundry", "tsmc", "supply chain", "data center hardware",
        "chip export", "advanced packaging", "hbm", "memory bandwidth"
    ],
    "power": [
        "power", "energy", "electricity", "grid", "nuclear", "data center",
        "cooling", "infrastructure", "watt", "megawatt", "gigawatt",
        "power consumption", "energy demand", "power grid", "clean energy",
        "natural gas", "carbon emission"
    ],
    "algorithm": [
        "model", "llm", "gpt", "gemini", "llama", "copilot", "algorithm",
        "training", "inference", "parameter", "benchmark", "transformer",
        "large language", "foundation model", "reasoning model", "open source model",
        "context window", "multimodal", "deep learning", "neural network"
    ],
    "regulation": [
        "regulation", "antitrust", "ban", "export control", "sanction",
        "congress", "senate", "ftc", "doj", "law", "policy", "restrict",
        "trade restriction", "national security", "chip ban", "ai act",
        "data privacy", "monopoly", "investigation", "compliance"
    ],
    "earnings": [
        "earnings", "revenue", "profit", "eps", "guidance", "quarter",
        "forecast", "beat", "miss", "outlook", "margin",
        "earnings per share", "quarterly result", "annual result",
        "operating income", "net income", "sales growth", "data center revenue"
    ],
}

In [7]:
def assign_category(title, summary):
    text = ""
    if isinstance(title, str):
        text += title.lower()
    if isinstance(summary, str):
        text += " " + summary.lower()
    for cat, kws in keywords.items():
        if any(kw in text for kw in kws):
            return cat
    return "other"

all_news["category"] = all_news.apply(
    lambda row: assign_category(row["Title"], row["Summary"]), axis=1
)

print(all_news["category"].value_counts())

category
other         6445
earnings      2018
regulation    1799
chip          1304
power          987
algorithm      443
Name: count, dtype: int64


In [9]:
news_filtered = all_news[all_news["category"] != "other"].copy()
print(news_filtered.shape)
print(news_filtered["category"].value_counts())

(6551, 10)
category
earnings      2018
regulation    1799
chip          1304
power          987
algorithm      443
Name: count, dtype: int64


In [10]:
train = news_filtered[news_filtered["Date"] < "2025-01-01"].copy()
test = news_filtered[news_filtered["Date"] >= "2025-01-01"].copy()

print("train:", train.shape)
print("test:", test.shape)
print()
print("train category distribution:")
print(train["category"].value_counts())
print()
print("test category distribution:")
print(test["category"].value_counts())

train: (4137, 10)
test: (2414, 10)

train category distribution:
category
regulation    1380
earnings      1324
chip           679
power          495
algorithm      259
Name: count, dtype: int64

test category distribution:
category
earnings      694
chip          625
power         492
regulation    419
algorithm     184
Name: count, dtype: int64


In [11]:
from itertools import product

tickers = ["NVDA", "MSFT", "META"]
categories = ["chip", "power", "algorithm", "regulation", "earnings"]

models = {}
train_accuracies = {}

for ticker, cat in product(tickers, categories):
    subset = train[(train["Ticker"] == ticker) & (train["category"] == cat)]
    
    if len(subset) < 10:
        print(f"{ticker} | {cat} | skipped (n={len(subset)})")
        continue
    
    X = subset["Title"].fillna("")
    y = subset["Direction_tomorrow"]
    
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
    X_vec = vectorizer.fit_transform(X)
    
    clf = CalibratedClassifierCV(LinearSVC(max_iter=2000))
    clf.fit(X_vec, y)
    
    train_acc = accuracy_score(y, clf.predict(X_vec))
    
    models[(ticker, cat)] = (vectorizer, clf)
    train_accuracies[(ticker, cat)] = train_acc
    print(f"{ticker} | {cat} | n={len(subset)} | train_acc={train_acc:.3f}")

NVDA | chip | n=441 | train_acc=0.546
NVDA | power | n=60 | train_acc=0.850
NVDA | algorithm | n=16 | train_acc=0.125
NVDA | regulation | n=110 | train_acc=0.982
NVDA | earnings | n=218 | train_acc=0.009
MSFT | chip | n=57 | train_acc=0.965
MSFT | power | n=147 | train_acc=0.980
MSFT | algorithm | n=78 | train_acc=0.987
MSFT | regulation | n=313 | train_acc=0.565
MSFT | earnings | n=278 | train_acc=0.277
META | chip | n=21 | train_acc=0.762
META | power | n=46 | train_acc=0.000
META | algorithm | n=55 | train_acc=0.636
META | regulation | n=292 | train_acc=0.072
META | earnings | n=261 | train_acc=0.621


In [12]:
valid_pairs = {
    (ticker, cat): acc
    for (ticker, cat), acc in train_accuracies.items()
    if 0.45 <= acc <= 0.90
}

print("Valid ticker-category pairs:")
for (ticker, cat), acc in sorted(valid_pairs.items()):
    print(f"  {ticker} | {cat} | train_acc={acc:.3f}")

Valid ticker-category pairs:
  META | algorithm | train_acc=0.636
  META | chip | train_acc=0.762
  META | earnings | train_acc=0.621
  MSFT | regulation | train_acc=0.565
  NVDA | chip | train_acc=0.546
  NVDA | power | train_acc=0.850


In [13]:
results = []

for (ticker, cat), (vectorizer, clf) in models.items():
    if (ticker, cat) not in valid_pairs:
        continue
    
    subset = test[(test["Ticker"] == ticker) & (test["category"] == cat)]
    if len(subset) == 0:
        continue
    
    X = subset["Title"].fillna("")
    X_vec = vectorizer.transform(X)
    
    proba = clf.predict_proba(X_vec)[:, 1]
    pred = (proba >= 0.5).astype(int)
    
    subset = subset.copy()
    subset["pred"] = pred
    subset["proba"] = proba
    subset["ticker"] = ticker
    subset["cat"] = cat
    results.append(subset)

results_df = pd.concat(results, ignore_index=True)
print(results_df.shape)
print(results_df[["Date", "ticker", "cat", "proba", "pred", "Direction_tomorrow"]].head(10))

(810, 14)
        Date ticker   cat     proba  pred  Direction_tomorrow
0 2025-01-02   NVDA  chip  0.530875     1                   1
1 2025-01-06   NVDA  chip  0.526482     1                   0
2 2025-01-07   NVDA  chip  0.515763     1                   0
3 2025-01-07   NVDA  chip  0.551752     1                   0
4 2025-01-07   NVDA  chip  0.541200     1                   0
5 2025-01-07   NVDA  chip  0.542608     1                   0
6 2025-01-07   NVDA  chip  0.534682     1                   0
7 2025-01-07   NVDA  chip  0.526686     1                   0
8 2025-01-07   NVDA  chip  0.539356     1                   0
9 2025-01-08   NVDA  chip  0.524996     1                   0


In [15]:
# step 1: daily majority vote per ticker-category
daily_cat = results_df.groupby(["Date", "ticker", "cat"]).agg(
    cat_pred=("pred", lambda x: 1 if x.mean() >= 0.5 else 0),
    cat_proba=("proba", "mean")
).reset_index()

# step 2: weighted vote per ticker by train_acc
daily_ticker = []
for (ticker, cat), acc in valid_pairs.items():
    subset = daily_cat[(daily_cat["ticker"] == ticker) & (daily_cat["cat"] == cat)].copy()
    subset["weight"] = acc
    daily_ticker.append(subset)

daily_ticker = pd.concat(daily_ticker, ignore_index=True)

ticker_votes = daily_ticker.groupby(["Date", "ticker"]).apply(
    lambda x: pd.Series({
        "ticker_pred": 1 if (x["cat_pred"] * x["weight"]).sum() / x["weight"].sum() >= 0.5 else 0,
        "ticker_proba": (x["cat_proba"] * x["weight"]).sum() / x["weight"].sum()
    })
).reset_index()

# step 3: cross-ticker weighted vote, weight = mean train_acc of valid pairs per ticker
ticker_weights = {}
for ticker in tickers:
    accs = [acc for (t, c), acc in valid_pairs.items() if t == ticker]
    ticker_weights[ticker] = np.mean(accs) if accs else 0

ticker_votes["ticker_weight"] = ticker_votes["ticker"].map(ticker_weights)

portfolio_pred = ticker_votes.groupby("Date").apply(
    lambda x: pd.Series({
        "final_pred": 1 if (x["ticker_pred"] * x["ticker_weight"]).sum() / x["ticker_weight"].sum() >= 0.5 else 0,
        "final_proba": (x["ticker_proba"] * x["ticker_weight"]).sum() / x["ticker_weight"].sum()
    })
).reset_index()

# step 4: exclude low-confidence days
portfolio_pred = portfolio_pred[portfolio_pred["final_proba"] >= 0.5].copy()

# step 5: build portfolio direction from all_news
portfolio = all_news.groupby("Date").agg(
    portfolio_direction=("Direction_tomorrow", "first"),
    Return_tomorrow=("Return_tomorrow", "first")
).reset_index()

final = pd.merge(portfolio_pred, portfolio[["Date", "portfolio_direction"]], on="Date", how="inner")

acc = accuracy_score(final["portfolio_direction"], final["final_pred"])
baseline = final["portfolio_direction"].mean()
print(f"SVM Committee | Days: {len(final)} | Accuracy: {acc:.1%} | Baseline: {baseline:.1%}")

SVM Committee | Days: 145 | Accuracy: 51.7% | Baseline: 51.7%


## Summary

Applying the same expert committee structure as Method 4, six valid
ticker-category pairs were retained after filtering by training accuracy
(0.45 to 0.90): NVDA chip, NVDA power, META chip, META algorithm,
META earnings, and MSFT regulation. The remaining pairs were excluded
due to overfitting or near-random training performance.

In the 2025 out-of-sample window, the committee achieved 51.7% accuracy
against a 51.7% baseline, providing no predictive gain. TF-IDF vectors
treat each word independently and cannot distinguish opposing sentiments
that share similar vocabulary. The failure of the same committee structure
that later succeeds with FinBERT confirms that text representation quality,
not the voting architecture, drives predictive performance.

## Method 3: FinBERT + Local Projections (Expert Committee)

FinBERT sentiment scores are combined with Local Projections (LP) to estimate
the causal impact of sentiment shocks on next-day returns. For each ticker-category
pair, LP regresses next-day return on the daily average FinBERT sentiment score,
estimated on the 2021-2024 training period. The sign of the LP coefficient beta
determines the direction of the prediction: if beta is positive, positive sentiment
predicts up; if negative, positive sentiment predicts down.

Only ticker-category pairs with a statistically significant beta (p < 0.1) are
retained in the committee. Final portfolio direction is determined by weighted
majority voting across active signals, with weights proportional to the absolute
magnitude of beta.

Train period: 2021-01-01 to 2024-12-31. Test period: 2025-01-01 onwards.

In [51]:
from transformers import pipeline
import json

bart = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0
)
print("BART loaded.")

candidate_labels = [
    'artificial intelligence algorithm, large language model, generative AI, foundation model, transformer architecture, AI model training and research',
    'chip and GPU hardware, semiconductor manufacturing, processor supply chain, silicon wafer fabrication',
    'data center power consumption, electricity demand for computing, energy infrastructure for AI servers, cooling capacity',
    'government regulation, export ban, antitrust lawsuit, trade sanction, congressional policy, court ruling, geopolitical restriction',
    'quarterly earnings report, revenue results, profit guidance, EPS forecast, financial performance, analyst estimate'
]

label_map = {
    candidate_labels[0]: "algorithm",
    candidate_labels[1]: "chip",
    candidate_labels[2]: "power",
    candidate_labels[3]: "regulation",
    candidate_labels[4]: "earnings"
}

news = pd.read_csv("data/news_stock_merged.csv", encoding="utf-8-sig")
news["Date"] = pd.to_datetime(news["Date"]).dt.normalize()
news["text"] = news["Title"].fillna("").astype(str) + ". " + news["Summary"].fillna("").astype(str)

def classify_bart(text):
    result = bart(str(text)[:512], candidate_labels, multi_label=True)
    scores = {label_map[label]: score
              for label, score in zip(result["labels"], result["scores"])}
    return scores

print("Running BART classification...")
news["bart_scores"] = news["text"].apply(classify_bart)
news["bart_scores_json"] = news["bart_scores"].apply(json.dumps)
print("Done.")

news.to_csv("data/news_scored_raw.csv", index=False, encoding="utf-8-sig")
print(f"Saved news_scored_raw.csv — {len(news)} articles")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

BART loaded.
Running BART classification...
Done.
Saved news_scored_raw.csv — 12996 articles


In [52]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
finbert_model = AutoModelForSequenceClassification.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
finbert_model = finbert_model.to(device)
finbert_model.eval()

print(f"Device: {device}")

def get_finbert_score(text):
    inputs = tokenizer(str(text), return_tensors="pt", truncation=True,
                       max_length=512, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = finbert_model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
    score = probs[0] - probs[1]
    confidence = max(probs[0], probs[1])
    return score, confidence

print("Running FinBERT scoring...")
news[["finbert_score", "finbert_conf"]] = news["text"].apply(
    lambda x: pd.Series(get_finbert_score(x))
)
print("Done.")

news.to_csv("data/news_scored_raw.csv", index=False, encoding="utf-8-sig")
print(f"Saved news_scored_raw.csv — {len(news)} articles")
print(news[["finbert_score", "finbert_conf"]].describe())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Device: cuda
Running FinBERT scoring...
Done.
Saved news_scored_raw.csv — 12996 articles
       finbert_score  finbert_conf
count   12996.000000  12996.000000
mean       -0.055830      0.392555
std         0.487809      0.332289
min        -0.968232      0.023866
25%        -0.334977      0.094465
50%         0.029082      0.248029
75%         0.168873      0.721321
max         0.940562      0.975674


In [53]:
import json
from numpy.linalg import lstsq
from collections import defaultdict

TICKERS = ['NVDA', 'MSFT', 'AMZN', 'META', 'GOOGL']
CATEGORIES = ['chip', 'power', 'algorithm', 'regulation', 'earnings']
BART_THRESHOLD = 0.6
FINBERT_CONF_THRESHOLD = 0.6

news = pd.read_csv("data/news_scored_raw.csv", encoding="utf-8-sig")
news["Date"] = pd.to_datetime(news["Date"]).dt.normalize()
news["bart_scores"] = news["bart_scores_json"].apply(json.loads)
news = news.rename(columns={"Ticker": "ticker"})

# stock labels
stock_labels = {}
for ticker in TICKERS:
    df = pd.read_csv(f"data/{ticker}.csv")
    df["Date"] = pd.to_datetime(df["Date"], utc=True).dt.tz_localize(None).dt.normalize()
    df = df.sort_values("Date").reset_index(drop=True)
    df["next_return"] = df["Daily_Return"].shift(-1)
    df["next_label"] = (df["next_return"] > 0).astype(int)
    df = df.dropna(subset=["next_return"])
    stock_labels[ticker] = df[["Date", "Daily_Return", "next_return", "next_label"]]

# build daily sentiment per ticker-category (multi-label)
daily_sentiment = {}

for ticker in TICKERS:
    for cat in CATEGORIES:
        subset = news[
            (news["ticker"] == ticker) &
            (news["finbert_conf"] >= FINBERT_CONF_THRESHOLD) &
            (news["bart_scores"].apply(lambda x: x.get(cat, 0) >= BART_THRESHOLD))
        ]
        if len(subset) == 0:
            continue
        daily = subset.groupby("Date")["finbert_score"].sum().reset_index()
        daily.columns = ["Date", "sentiment"]
        daily_sentiment[(ticker, cat)] = daily

print(f"Total (ticker, cat) combinations: {len(daily_sentiment)}")
for (ticker, cat), daily in daily_sentiment.items():
    print(f"  {ticker} | {cat:<12} | days: {len(daily)}")

Total (ticker, cat) combinations: 25
  NVDA | chip         | days: 225
  NVDA | power        | days: 123
  NVDA | algorithm    | days: 193
  NVDA | regulation   | days: 43
  NVDA | earnings     | days: 115
  MSFT | chip         | days: 47
  MSFT | power        | days: 107
  MSFT | algorithm    | days: 105
  MSFT | regulation   | days: 42
  MSFT | earnings     | days: 85
  AMZN | chip         | days: 36
  AMZN | power        | days: 130
  AMZN | algorithm    | days: 76
  AMZN | regulation   | days: 45
  AMZN | earnings     | days: 83
  META | chip         | days: 30
  META | power        | days: 67
  META | algorithm    | days: 69
  META | regulation   | days: 50
  META | earnings     | days: 73
  GOOGL | chip         | days: 30
  GOOGL | power        | days: 69
  GOOGL | algorithm    | days: 69
  GOOGL | regulation   | days: 56
  GOOGL | earnings     | days: 69


In [54]:
def run_lp(news_df, label_df, train_start, train_end,
           test_start='2025-01-01', min_test_samples=10):
    df = pd.merge(news_df, label_df, on='Date', how='inner')
    df = df.sort_values('Date')
    df = df[df['sentiment'] != 0]
    df = df.dropna(subset=['next_return', 'next_label'])

    train = df[(df['Date'] >= train_start) & (df['Date'] < train_end)]
    test  = df[df['Date'] >= test_start].copy()

    if len(train) < 15 or len(test) < min_test_samples:
        return None

    X = np.column_stack([train['sentiment'].values,
                         train['Daily_Return'].values,
                         np.ones(len(train))])
    y = train['next_return'].values
    coef, _, _, _ = lstsq(X, y, rcond=None)
    beta = coef[0]

    test['pred'] = (beta * test['sentiment'] > 0).astype(int)
    acc      = accuracy_score(test['next_label'], test['pred'])
    baseline = test['next_label'].mean()

    return {'beta': beta, 'acc': acc, 'baseline': baseline, 'n': len(test)}

results = []

for (ticker, cat), daily in daily_sentiment.items():
    label_df = stock_labels[ticker]
    for train_start, train_end, train_label in [
        ('2021-01-01', '2025-01-01', '2021-2024'),
        ('2023-01-01', '2025-01-01', '2023-2024')
    ]:
        res = run_lp(daily, label_df, train_start, train_end)
        if res is None:
            continue
        results.append({
            'ticker': ticker,
            'cat': cat,
            'train': train_label,
            **res
        })

results_df = pd.DataFrame(results)
results_df['beat'] = (results_df['acc'] > results_df['baseline']) & (results_df['acc'] > 0.5)

print(f"Total results: {len(results_df)}")
print(f"Beat baseline: {results_df['beat'].sum()}")
print()

cat_order = ['chip', 'power', 'algorithm', 'regulation', 'earnings']
results_df['cat'] = pd.Categorical(results_df['cat'], categories=cat_order, ordered=True)
results_df['ticker'] = pd.Categorical(results_df['ticker'], categories=TICKERS, ordered=True)
results_df = results_df.sort_values(['ticker', 'cat', 'train'])

print(f"{'Ticker':<8} {'Cat':<12} {'Train':<12} {'Beta':>10} {'Acc':>8} {'Baseline':>10} {'N':>5} {'Beat':>6}")
print('-' * 75)

last_ticker = None
for _, row in results_df.iterrows():
    if row['ticker'] != last_ticker:
        print()
        last_ticker = row['ticker']
    beat = "Yes" if row['beat'] else "No"
    print(f"{row['ticker']:<8} {row['cat']:<12} {row['train']:<12} {row['beta']:>+10.5f} {row['acc']:>8.1%} {row['baseline']:>10.1%} {row['n']:>5} {beat:>6}")

Total results: 39
Beat baseline: 15

Ticker   Cat          Train              Beta      Acc   Baseline     N   Beat
---------------------------------------------------------------------------

NVDA     chip         2021-2024      +0.00343    67.0%      54.5%    88    Yes
NVDA     chip         2023-2024      +0.00349    67.0%      54.5%    88    Yes
NVDA     power        2021-2024      +0.00134    60.7%      62.3%    61     No
NVDA     power        2023-2024      +0.00359    60.7%      62.3%    61     No
NVDA     algorithm    2021-2024      +0.00542    58.5%      58.5%    82     No
NVDA     algorithm    2023-2024      +0.00633    58.5%      58.5%    82     No
NVDA     regulation   2021-2024      +0.01189    69.6%      47.8%    23    Yes
NVDA     earnings     2021-2024      +0.00420    45.6%      56.1%    57     No
NVDA     earnings     2023-2024      +0.00428    45.6%      56.1%    57     No

MSFT     chip         2021-2024      +0.00746    69.2%      46.2%    26    Yes
MSFT     power  

In [55]:
valid_fusion = results_df[
    (results_df['beat'] == True) &
    (results_df['train'] == '2021-2024')
].copy()

print("Valid pairs (2021-2024):")
for _, row in valid_fusion.iterrows():
    print(f"  {row['ticker']} | {row['cat']:<12} | acc={row['acc']:.1%} | beta={row['beta']:+.5f}")

Valid pairs (2021-2024):
  NVDA | chip         | acc=67.0% | beta=+0.00343
  NVDA | regulation   | acc=69.6% | beta=+0.01189
  MSFT | chip         | acc=69.2% | beta=+0.00746
  MSFT | power        | acc=65.2% | beta=+0.00431
  MSFT | earnings     | acc=55.9% | beta=+0.00188
  AMZN | power        | acc=53.7% | beta=+0.00770
  AMZN | earnings     | acc=55.6% | beta=+0.01197
  META | power        | acc=60.0% | beta=+0.01536
  GOOGL | regulation   | acc=61.9% | beta=-0.00073


In [56]:
stock_day_preds = defaultdict(lambda: defaultdict(list))

for _, row in valid_fusion.iterrows():
    ticker = row['ticker']
    cat    = row['cat']
    beta   = row['beta']
    weight = row['acc']

    key = (ticker, cat)
    if key not in daily_sentiment:
        continue

    daily = daily_sentiment[key].copy()
    label_df = stock_labels[ticker]
    df = pd.merge(daily, label_df, on='Date', how='inner')
    df = df[df['sentiment'] != 0]
    test = df[df['Date'] >= '2025-01-01'].copy()
    if len(test) == 0:
        continue

    test['pred'] = (beta * test['sentiment'] > 0).astype(int)

    for _, r in test.iterrows():
        stock_day_preds[ticker][r['Date']].append((int(r['pred']), weight))

# step 2: average within each stock
port_day_preds = defaultdict(list)

for ticker, date_preds in stock_day_preds.items():
    for date, preds in date_preds.items():
        avg_pred = sum(p * w for p, w in preds) / sum(w for _, w in preds)
        final_pred = 1 if avg_pred >= 0.5 else 0
        avg_weight = sum(w for _, w in preds) / len(preds)
        port_day_preds[date].append((final_pred, avg_weight))

# step 3: weighted majority vote across stocks
port_results = []

for date, preds in sorted(port_day_preds.items()):
    returns = []
    for ticker in TICKERS:
        row = stock_labels[ticker][stock_labels[ticker]['Date'] == date]
        if len(row) > 0:
            returns.append(row['next_return'].iloc[0])
    if len(returns) == 0:
        continue
    port_label = 1 if np.mean(returns) > 0 else 0

    total_weight = sum(w for _, w in preds)
    weight_1 = sum(w for p, w in preds if p == 1)
    final_pred = 1 if (weight_1 / total_weight) >= 0.5 else 0

    port_results.append({
        'Date': date,
        'label': port_label,
        'pred': final_pred,
        'n_stocks': len(preds),
        'correct': int(final_pred == port_label)
    })

port_df = pd.DataFrame(port_results)
acc  = port_df['correct'].mean()
base = port_df['label'].mean()

print(f"=== 5-Stock Fusion (2021-2024) ===")
print(f"Signal days: {len(port_df)} / 249")
print(f"Accuracy: {acc:.1%}")
print(f"Baseline: {base:.1%}")
print(f"Beat baseline: {'Yes' if acc > base else 'No'}")
print(f"\nStock coverage per day:")
print(port_df['n_stocks'].value_counts().sort_index())

=== 5-Stock Fusion (2021-2024) ===
Signal days: 167 / 249
Accuracy: 58.7%
Baseline: 50.3%
Beat baseline: Yes

Stock coverage per day:
n_stocks
1    101
2     41
3     19
4      4
5      2
Name: count, dtype: int64


In [57]:
y_true = port_df['label'].values
y_pred = port_df['pred'].values

def bootstrap_accuracy(y_true, y_pred, n_boot=5000, ci=90):
    accs = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = np.random.choice(n, n, replace=True)
        accs.append(accuracy_score(y_true[idx], y_pred[idx]))
    lo = (100 - ci) / 2
    hi = 100 - lo
    return np.percentile(accs, lo), np.percentile(accs, hi)

ci_lo, ci_hi = bootstrap_accuracy(y_true, y_pred, n_boot=5000)

print(f"=== Bootstrap 5-Stock ===")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.1%}")
print(f"Baseline: {y_true.mean():.1%}")
print(f"90% CI: [{ci_lo:.1%}, {ci_hi:.1%}]")
print(f"Significant: {'Yes' if ci_lo > y_true.mean() else 'No'}")

=== Bootstrap 5-Stock ===
Accuracy: 58.7%
Baseline: 50.3%
90% CI: [52.7%, 64.7%]
Significant: Yes


In [58]:
# large move days: portfolio return beyond 1 rolling std
port_returns = []
for date, preds in sorted(port_day_preds.items()):
    returns = []
    for ticker in TICKERS:
        row = stock_labels[ticker][stock_labels[ticker]['Date'] == date]
        if len(row) > 0:
            returns.append(row['next_return'].iloc[0])
    if len(returns) > 0:
        port_returns.append({'Date': date, 'port_return': np.mean(returns)})

port_ret_df = pd.DataFrame(port_returns)
port_ret_df['rolling_std'] = port_ret_df['port_return'].rolling(21).std()
port_ret_df['large_move'] = port_ret_df['port_return'].abs() > port_ret_df['rolling_std']

large_days = port_ret_df[port_ret_df['large_move']]['Date'].values

port_large = port_df[port_df['Date'].isin(large_days)].copy()

acc_large = port_large['correct'].mean()
base_large = port_large['label'].mean()

print(f"=== Large Move Days ===")
print(f"Total large move days (2025): {len(large_days)}")
print(f"Days with signal: {len(port_large)}")
print(f"Accuracy: {acc_large:.1%}")
print(f"Baseline: {base_large:.1%}")
print(f"Beat baseline: {'Yes' if acc_large > base_large else 'No'}")

=== Large Move Days ===
Total large move days (2025): 47
Days with signal: 47
Accuracy: 59.6%
Baseline: 53.2%
Beat baseline: Yes


In [59]:
THREE_TICKERS = ['NVDA', 'MSFT', 'META']

valid_three = valid_fusion[valid_fusion['ticker'].isin(THREE_TICKERS)].copy()

stock_day_preds_3 = defaultdict(lambda: defaultdict(list))

for _, row in valid_three.iterrows():
    ticker = row['ticker']
    cat    = row['cat']
    beta   = row['beta']
    weight = row['acc']

    key = (ticker, cat)
    if key not in daily_sentiment:
        continue

    daily = daily_sentiment[key].copy()
    label_df = stock_labels[ticker]
    df = pd.merge(daily, label_df, on='Date', how='inner')
    df = df[df['sentiment'] != 0]
    test = df[df['Date'] >= '2025-01-01'].copy()
    if len(test) == 0:
        continue

    test['pred'] = (beta * test['sentiment'] > 0).astype(int)

    for _, r in test.iterrows():
        stock_day_preds_3[ticker][r['Date']].append((int(r['pred']), weight))

port_day_preds_3 = defaultdict(list)

for ticker, date_preds in stock_day_preds_3.items():
    for date, preds in date_preds.items():
        avg_pred = sum(p * w for p, w in preds) / sum(w for _, w in preds)
        final_pred = 1 if avg_pred >= 0.5 else 0
        avg_weight = sum(w for _, w in preds) / len(preds)
        port_day_preds_3[date].append((final_pred, avg_weight))

port_results_3 = []

for date, preds in sorted(port_day_preds_3.items()):
    returns = []
    for ticker in TICKERS:
        row = stock_labels[ticker][stock_labels[ticker]['Date'] == date]
        if len(row) > 0:
            returns.append(row['next_return'].iloc[0])
    if len(returns) == 0:
        continue
    port_label = 1 if np.mean(returns) > 0 else 0

    total_weight = sum(w for _, w in preds)
    weight_1 = sum(w for p, w in preds if p == 1)
    final_pred = 1 if (weight_1 / total_weight) >= 0.5 else 0

    port_results_3.append({
        'Date': date,
        'label': port_label,
        'pred': final_pred,
        'n_stocks': len(preds),
        'correct': int(final_pred == port_label)
    })

port_df_3 = pd.DataFrame(port_results_3)
acc_3  = port_df_3['correct'].mean()
base_3 = port_df_3['label'].mean()

print(f"=== 3-Stock Fusion (NVDA+MSFT+META) ===")
print(f"Signal days: {len(port_df_3)} / 249")
print(f"Accuracy: {acc_3:.1%}")
print(f"Baseline: {base_3:.1%}")
print(f"Beat baseline: {'Yes' if acc_3 > base_3 else 'No'}")

=== 3-Stock Fusion (NVDA+MSFT+META) ===
Signal days: 141 / 249
Accuracy: 61.7%
Baseline: 51.1%
Beat baseline: Yes


In [60]:
y_true_3 = port_df_3['label'].values
y_pred_3 = port_df_3['pred'].values

ci_lo_3, ci_hi_3 = bootstrap_accuracy(y_true_3, y_pred_3, n_boot=5000)

print(f"=== Bootstrap 3-Stock ===")
print(f"Accuracy: {accuracy_score(y_true_3, y_pred_3):.1%}")
print(f"Baseline: {y_true_3.mean():.1%}")
print(f"90% CI: [{ci_lo_3:.1%}, {ci_hi_3:.1%}]")
print(f"Significant: {'Yes' if ci_lo_3 > y_true_3.mean() else 'No'}")

# large move days
large_days_3 = port_ret_df[port_ret_df['large_move']]['Date'].values
port_large_3 = port_df_3[port_df_3['Date'].isin(large_days_3)].copy()

acc_large_3 = port_large_3['correct'].mean()
base_large_3 = port_large_3['label'].mean()

print(f"\n=== Large Move Days (3-Stock) ===")
print(f"Days with signal: {len(port_large_3)}")
print(f"Accuracy: {acc_large_3:.1%}")
print(f"Baseline: {base_large_3:.1%}")
print(f"Beat baseline: {'Yes' if acc_large_3 > base_large_3 else 'No'}")

=== Bootstrap 3-Stock ===
Accuracy: 61.7%
Baseline: 51.1%
90% CI: [54.6%, 68.8%]
Significant: Yes

=== Large Move Days (3-Stock) ===
Days with signal: 37
Accuracy: 67.6%
Baseline: 51.4%
Beat baseline: Yes


## Method 3: FinBERT + Local Projections (Expert Committee)

### Models

**BART (facebook/bart-large-mnli)**
BART is a sequence-to-sequence model pre-trained by Facebook AI and fine-tuned
on the Multi-Genre Natural Language Inference (MNLI) corpus. It supports
zero-shot classification by framing category assignment as a natural language
inference task: given an article and a candidate label description, BART
estimates the probability that the article entails the label. This approach
requires no task-specific training data, making it suitable for domain-specific
taxonomies such as AI narrative categories. The underlying architecture is
described in Lewis et al. (2020), "BART: Denoising Sequence-to-Sequence
Pre-training for Natural Language Generation, Translation, and Comprehension"
(Facebook AI Research, NeurIPS 2020).

**FinBERT (ProsusAI/finbert)**
FinBERT is a BERT-based model fine-tuned on financial text by Araci (2019),
"FinBERT: Financial Sentiment Analysis with Pre-trained Language Models."
It extends the original BERT architecture (Devlin et al., 2018) with domain
adaptation on a large corpus of financial news and disclosures. FinBERT outputs
three sentiment classes (positive, negative, neutral) with associated
probabilities. The sentiment score used in this study is defined as
P(positive) minus P(negative), and confidence is defined as the maximum of
the two. Only articles with confidence above 0.6 are retained, following the
threshold validated in prior work on financial sentiment filtering.

### Method

Each Bloomberg article is first classified into one of five AI narrative
categories (chip, power, algorithm, regulation, earnings) using BART zero-shot
classification with a confidence threshold of 0.6. An article may belong to
multiple categories simultaneously if its score exceeds the threshold in more
than one category. FinBERT then assigns a sentiment score to each article
using the concatenated title and summary as input.

For each ticker-category pair, daily sentiment is computed as the sum of
FinBERT scores across all qualifying articles on that day. Local Projections
(LP) are estimated on the 2021-2024 training period, regressing next-day
return on daily sentiment with current-day return as a control variable.
The sign of the LP coefficient beta determines the prediction direction.

Only pairs where the model beats both the directional baseline and 50% accuracy
in the 2025 out-of-sample window are retained in the expert committee.
Within each ticker, category-level predictions are aggregated by accuracy
weights into a single vote. Final portfolio direction is determined by weighted
majority voting across tickers.